In [1]:
import pandas as pd
import nltk
import re
import matplotlib.pyplot as plt
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
import spacy
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, TensorDataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
nlp = spacy.load("en_core_web_sm")

news_df = pd.read_csv('/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv')
df = news_df
news_df.head()

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83


This section of codes belong to "moreCleaned_Emotion_Detection,_HLT_project.ipynb"

In [4]:
# Handling missing values
news_df.fillna('', inplace=True)

# Removing duplicate rows
news_df = news_df.drop_duplicates()

# Converting to lowercase
news_df['title'] = news_df['title'].str.lower()
news_df['text'] = news_df['text'].str.lower()

# Removing punctuation
news_df['title'] = news_df['title'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
news_df['text'] = news_df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# Removing titles with less than 3 word, texts with less than 50 words
news_df = news_df[news_df['title'].apply(lambda x: len(x.split()) >= 3)]
news_df = news_df[news_df['text'].apply(lambda x: len(x.split()) >= 50)]

In [5]:
def tokenize_and_lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

news_df['titleTokens'] = news_df['title'].apply(tokenize_and_lemmatize)
news_df['textTokens'] = news_df['text'].apply(tokenize_and_lemmatize)

In [6]:
def load_lexicon(filepath):
    lexicon = {}
    with open(filepath, 'r') as file:
        for line in file:
            word, valence, arousal, dominance = line.strip().split('\t')
            lexicon[word.lower()] = {'valence': float(valence), 'arousal': float(arousal), 'dominance': float(dominance)}
    return lexicon

lexicon = load_lexicon('/content/drive/MyDrive/HLT project/Project files/NRC-VAD-Lexicon.txt')

In [7]:
list(lexicon)[:5]

['aaaaaaah', 'aaaah', 'aardvark', 'aback', 'abacus']

# Let's use FinBert!

Without any training on our data:


In [10]:
# Use a pipeline as a high-level helper
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
torch.device(device)

pipe = pipeline("text-classification", model="ProsusAI/finbert", truncation=True)

cuda


In [11]:
import torch

x = []
for i in news_df.index:
  x.append(pipe(" ".join(news_df['titleTokens'][i])))

In [12]:
filtered_x = [el[0]['label'] for el in x]

In [13]:
predictions = filtered_x
targets = list(news_df['true_sentiment'].apply(lambda a : a.lower()))
len(x), len(predictions), len(targets), len(news_df.index)

(2274, 2274, 2274, 2274)

In [14]:
print(classification_report(targets, predictions, labels=["positive", "negative", "neutral"]))

              precision    recall  f1-score   support

    positive       0.61      0.60      0.60       765
    negative       0.57      0.65      0.61       709
     neutral       0.52      0.46      0.49       800

    accuracy                           0.57      2274
   macro avg       0.57      0.57      0.57      2274
weighted avg       0.56      0.57      0.56      2274



In [ ]:
import torch

x = []
for i in news_df.index:
  with torch.no_grad():
    a = " ".join(news_df['textTokens'][i])
    try:
      x.append(pipe(a))
    except Exception as err:
      print("\n\n" + str(len(news_df['textTokens'][i])) + ' ' + str(err) + "\n\n")

predictions = [el[0]['label'] for el in x]
targets = list(news_df['true_sentiment'].apply(lambda a : a.lower()))

print(classification_report(targets, predictions, labels=["positive", "negative", "neutral"]))

With 80/20 split and fine-tune the model

In [22]:
from transformers import AdamW, BertTokenizer, AutoModelForSequenceClassification

convert_dict = {
    "positive": 0,
    "negative": 1,
    "neutral": 2
}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
torch.device(device)
torch.cuda.device(0)
print(torch.cuda.get_device_name(0))
model_name = "ProsusAI/finbert"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

texts = [" ".join(el) for el in list(news_df["titleTokens"])]
labels = [convert_dict[el.lower()] for el in list(news_df["true_sentiment"])]

tokenized_texts = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# Convert labels to tensors
labels = torch.tensor(labels)
split_text = list(zip(tokenized_texts["input_ids"], tokenized_texts['token_type_ids'], tokenized_texts['attention_mask']))
# Split data into train and validation sets
train_inputs, validation_inputs, train_labels, validation_labels = train_test_split([i for i in range(len(labels))], torch.Tensor(labels), test_size=0.2, random_state=42)

train_inputs = {
    "input_ids": tokenized_texts['input_ids'][train_inputs],
    "attention_mask": tokenized_texts['attention_mask'][train_inputs],
    "token_type_ids": tokenized_texts['token_type_ids'][train_inputs]
}

validation_inputs = {
    "input_ids": tokenized_texts['input_ids'][validation_inputs],
    "attention_mask": tokenized_texts['attention_mask'][validation_inputs],
    "token_type_ids": tokenized_texts['token_type_ids'][validation_inputs]
}
# print(b.size())
# Create DataLoader for training and validation sets
train_dataset = TensorDataset(train_inputs['input_ids'], train_inputs['attention_mask'], train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

validation_dataset = TensorDataset(validation_inputs['input_ids'], validation_inputs['attention_mask'], validation_labels)
validation_dataloader = DataLoader(validation_dataset, batch_size=4, shuffle=False)

# Define optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
min_loss = 1000.0
# Training loop
epochs = 3
for epoch in range(epochs):
    model.train()
    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in validation_dataloader:
            input_ids, attention_mask, labels = batch
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    if val_loss < min_loss:
      model.save_pretrained("/content/drive/MyDrive/HLT project/Models/finbert_model")
      min_loss = val_loss
    print(f"Epoch {epoch+1}/{epochs}, Validation Loss: {val_loss}, Accuracy: {accuracy}")

# Save the fine-tuned model

cuda
Tesla T4


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/3, Validation Loss: 117.38743035495281, Accuracy: 0.5824175824175825
Epoch 2/3, Validation Loss: 81.6501661464572, Accuracy: 0.7230769230769231
Epoch 3/3, Validation Loss: 105.7927446514368, Accuracy: 0.6615384615384615


In [24]:
model.eval()
predictions = []
true_labels = []

for batch in validation_dataloader:
  with torch.no_grad():
    input_ids, attention_mask, labels = batch
    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    _, predicted = torch.max(outputs.logits, 1)
    predictions.extend(predicted.tolist())
    true_labels.extend(labels.tolist())

print("Classification Report:")
print(classification_report(true_labels, predictions))

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       155
           1       0.56      0.91      0.69       139
           2       0.83      0.34      0.48       161

    accuracy                           0.66       455
   macro avg       0.71      0.67      0.64       455
weighted avg       0.72      0.66      0.64       455

